In [13]:
import torch
import torch.nn as nn

In [14]:
# input & output initialization
x = torch.tensor([
    [1, 2, 3, 4, 5],
    [2, 3, 4, 5, 6],
    [3, 4, 5, 6, 7],
    [1, 3, 5, 7, 9],
    [0, 2, 4, 6, 8],
    [4, 5, 6, 7, 8],
    [5, 4, 3, 2, 1],
    [6, 5, 4, 3, 2],
    [9, 7, 5, 3, 1],
    [3, 1, 4, 1, 5],
    [2, 7, 1, 8, 2],
    [4, 4, 4, 4, 4],
], dtype=torch.float32)
# 1 if increasing, 0 if decreasing (mixed or flat also count as 0)
y = torch.tensor([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0], dtype=torch.float32).view(-1, 1)
neurons = 4

In [15]:
# model declaration
class MultiLayerPerceptron(nn.Module):
    def __init__(self, input_size, output_size, hidden_size):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_features=input_size, out_features=hidden_size),
            nn.ReLU(),
            nn.Linear(in_features=hidden_size, out_features=output_size),
        )

    def forward(self, input):
        return self.head(input)

In [16]:
# model initialization
model = MultiLayerPerceptron(input_size=x.shape[-1], output_size=y.shape[-1], hidden_size=neurons)
for name, param in model.named_parameters():
    print(f"{name}: {param.data}")
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

head.0.weight: tensor([[ 0.4131, -0.0969, -0.3192,  0.1222,  0.0792],
        [ 0.3482, -0.4193,  0.2704, -0.2129, -0.2347],
        [-0.0349, -0.0696, -0.0457,  0.1117, -0.2826],
        [-0.1729, -0.1968,  0.2695,  0.3790,  0.1952]])
head.0.bias: tensor([-0.1183,  0.1624,  0.4189,  0.2384])
head.2.weight: tensor([[ 0.2120,  0.4391, -0.1793,  0.1951]])
head.2.bias: tensor([0.3166])


In [17]:
# training
epochs = 500
for epoch in range(epochs):
    # forward pass
    z = model(x)
    # calculate loss
    loss = nn.BCEWithLogitsLoss()(input=z, target=y)
    # backward pass
    optimizer.zero_grad()
    loss.backward()
    if epoch % 50 == 0:
        print(f"epoch {epoch} loss: {loss.item()}")
        for name, param in model.named_parameters():
            print(f"{name}.grad: {param.grad}")
    # update parameters
    optimizer.step()

epoch 0 loss: 0.7532361149787903
head.0.weight.grad: tensor([[ 0.3219,  0.2912,  0.1862,  0.1678,  0.0757],
        [ 0.5984,  0.4412,  0.4165,  0.2328,  0.2346],
        [-0.0212, -0.0744, -0.0106, -0.0850, -0.0212],
        [ 0.1843,  0.1661,  0.0796,  0.0728, -0.0019]])
head.0.bias.grad: tensor([ 0.0573,  0.1035, -0.0106,  0.0337])
head.2.weight.grad: tensor([[ 0.3069,  0.1098,  0.0086, -0.0403]])
head.2.bias.grad: tensor([0.2346])
epoch 50 loss: 0.16145949065685272
head.0.weight.grad: tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-0.0797, -0.0217, -0.0310,  0.0422,  0.0010],
        [ 0.0647,  0.0091,  0.0215, -0.0494, -0.0048]])
head.0.bias.grad: tensor([ 0.0000,  0.0000, -0.0203,  0.0176])
head.2.weight.grad: tensor([[ 0.0000,  0.0000,  0.0911, -0.1095]])
head.2.bias.grad: tensor([0.0309])
epoch 100 loss: 0.05515361949801445
head.0.weight.grad: tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [

In [18]:
# prediction
with torch.inference_mode():
    z = model(x)
# activation function (value > 0 is 1, else 0)
y_pred = (z > 0).float()
print(f"y_pred: {y_pred.view(-1)}")
print(f"y:      {y.view(-1)}")

y_pred: tensor([1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.])
y:      tensor([1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.])
